In [ ]:
# transformers:
#   Hugging Face 模型和 Trainer
# peft:
#   LoRA / QLoRA
# bitsandbytes:
#   4-bit quantization
# accelerate:
#   GPU / mixed precision / device management
# datasets:
#   Hugging Face Dataset
# scikit-learn:
#   train/validation split + metrics
# ============================================================

!pip install -q --no-cache-dir \
    "transformers>=4.45" \
    "peft>=0.13" \
    "accelerate>=0.34" \
    "bitsandbytes>=0.43" \
    "datasets>=2.20" \
    "scikit-learn==1.8.0" \
    "sentencepiece"


# # ============================================================
# # 修复 scikit-learn 版本混装问题
# # ============================================================

# # 先卸载当前 sklearn
# !pip uninstall -y scikit-learn

# # 安装与 Kaggle 当前 Python 3.12 环境兼容的稳定版本
# !pip install -q --no-cache-dir scikit-learn==1.8.0


# # ============================================================
# # 检查 scikit-learn 是否修复
# # ============================================================

# import sklearn

# print("scikit-learn version:", sklearn.__version__)

# from sklearn.model_selection import train_test_split
# from sklearn.metrics import (
#     log_loss,
#     accuracy_score,
#     f1_score,
#     confusion_matrix
# )

# print("scikit-learn import: OK")


In [ ]:
import os
import gc
import random
import warnings

import numpy as np
import pandas as pd

import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    log_loss,
    accuracy_score,
    f1_score,
    confusion_matrix,
)

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

warnings.filterwarnings("ignore")


# ============================================================
# Random Seed
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# Paths
# ============================================================

DATA_DIR = "/kaggle/input/competitions/llm-classification-finetuning"

TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")

OUTPUT_DIR = "/kaggle/working/qwen-qlora"


# ============================================================
# Model
# ============================================================

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"


# ============================================================
# Training Configuration
# ============================================================

# T4 16GB / P100 16GB 都可以先使用 1024,如果你的样本非常长，可以进一步改成 2048。
# 但是：prompt + response_a + response_b很容易产生很长的 sequence。
# Sequence 越长，显存消耗越大。
MAX_LENGTH = 1024

# 每个 GPU 每次只处理 2 个样本。
# 使用 gradient accumulation 模拟更大的 batch。
PER_DEVICE_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4

# 训练轮数
NUM_EPOCHS = 1

# LoRA learning rate
LEARNING_RATE = 2e-5

# ============================================================
# GPU information
# ============================================================

print("=" * 60)
print("CONFIGURATION")
print("=" * 60)

print("Model:", MODEL_NAME)
print("Max length:", MAX_LENGTH)
print("Epochs:", NUM_EPOCHS)
print("Batch size:", PER_DEVICE_BATCH_SIZE)
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)

print()

if torch.cuda.is_available():
    print("CUDA:", torch.version.cuda)
    print("GPU count:", torch.cuda.device_count())

    for i in range(torch.cuda.device_count()):
        print(
            f"GPU {i}:",
            torch.cuda.get_device_name(i)
        )
else:
    print("WARNING: CUDA is not available!")


# # ============================================================
# # Cell 4: Load Dataset
# # ============================================================

# train_df = pd.read_csv(TRAIN_PATH)
# test_df = pd.read_csv(TEST_PATH)

# print("=" * 60)
# print("DATASET")
# print("=" * 60)

# print("Train shape:", train_df.shape)
# print("Test shape :", test_df.shape)

# print()
# print("Train columns:")
# print(train_df.columns.tolist())
# display(train_df.head())

# print()
# print("Test columns:")
# print(test_df.columns.tolist())
# display(test_df.head())


In [ ]:
def get_label(row):
    """
    将 Kaggle 的三个 binary winner columns
    转换成一个三分类 label。

    label:
        0 -> model A wins
        1 -> model B wins
        2 -> tie
    """

    if row["winner_model_a"] == 1: return 0
    elif row["winner_model_b"] == 1: return 1
    elif row["winner_tie"] == 1: return 2
    else: return -1 # 正常情况下不应该出现
        
train_df["label"] = train_df.apply(
    get_label,
    axis=1
)


# 检查非法 label
invalid_count = (train_df["label"] == -1).sum()
print("Invalid labels:", invalid_count)


# 删除非法数据
if invalid_count > 0:
    train_df = train_df[
        train_df["label"] != -1
    ].reset_index(drop=True)

# ============================================================
# Class Distribution
# ============================================================

label_names = {
    0: "model_a",
    1: "model_b",
    2: "tie",
}

label_counts = train_df["label"].value_counts().sort_index()

print("=" * 60)
print("CLASS DISTRIBUTION")
print("=" * 60)

for label, count in label_counts.items():

    ratio = count / len(train_df)

    print(
        f"{label_names[label]:10s} "
        f"{count:8d} "
        f"{ratio:.4f}"
    )


# # ============================================================
# # Cell 6: Text Length Analysis
# # ============================================================

# train_df["prompt_chars"] = (
#     train_df["prompt"]
#     .fillna("")
#     .astype(str)
#     .str.len()
# )

# train_df["response_a_chars"] = (
#     train_df["response_a"]
#     .fillna("")
#     .astype(str)
#     .str.len()
# )

# train_df["response_b_chars"] = (
#     train_df["response_b"]
#     .fillna("")
#     .astype(str)
#     .str.len()
# )

# train_df["total_chars"] = (
#     train_df["prompt_chars"]
#     + train_df["response_a_chars"]
#     + train_df["response_b_chars"]
# )


# print(
#     train_df[
#         [
#             "prompt_chars",
#             "response_a_chars",
#             "response_b_chars",
#             "total_chars"
#         ]
#     ].describe()
# )


In [ ]:
def build_text(row):
    """
    将一个 Kaggle sample 转换成 LLM 输入。

    注意：
    不使用 model_a / model_b identity。

    原因：
    train 有模型身份，而 test 没有。
    使用模型身份会造成严重的数据分布不一致。
    """

    prompt = str(row["prompt"])
    response_a = str(row["response_a"])
    response_b = str(row["response_b"])

    text = (
        "You are an expert judge evaluating two AI assistant "
        "responses.\n\n"

        "Your task is to determine which response is better "
        "for the given user prompt.\n\n"

        "Consider correctness, relevance, helpfulness, "
        "clarity, completeness, and adherence to the user's "
        "instructions.\n\n"

        "There are three possible outcomes:\n"
        "A = Response A is better\n"
        "B = Response B is better\n"
        "TIE = Both responses are approximately equally good\n\n"

        "### User Prompt\n"
        f"{prompt}\n\n"

        "### Response A\n"
        f"{response_a}\n\n"

        "### Response B\n"
        f"{response_b}\n\n"

        "### Judgment\n"
    )

    return text

train_df["text"] = train_df.apply(build_text,axis=1)
test_df["text"] = test_df.apply(build_text,axis=1)
# print(train_df["text"].iloc[0][:5000])


In [ ]:
base_train, val_df = train_test_split(
    train_df,
    test_size=0.10,
    random_state=SEED,
    stratify=train_df["label"],
)

base_train = base_train.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Original train:", len(base_train))
print("Validation    :", len(val_df))


In [ ]:
def swap_row(row):
    """
    将 Response A 和 Response B 交换。

    同时修改 label：

        A win (0) -> B win (1)
        B win (1) -> A win (0)
        Tie   (2) -> Tie (2)

    这样可以显著减少模型学习：
        "第一个回答通常更好"
    这种 position bias。
    """

    new_row = row.copy()

    # Swap responses
    new_row["response_a"] = row["response_b"]
    new_row["response_b"] = row["response_a"]

    # Swap label
    if row["label"] == 0:
        new_row["label"] = 1

    elif row["label"] == 1:
        new_row["label"] = 0

    else:
        new_row["label"] = 2

    return new_row

# ============================================================
# Generate Swapped Training Data
# ============================================================

swap_train = base_train.apply(
    swap_row,
    axis=1
)

# 重新构造 text
swap_train["text"] = swap_train.apply(build_text,axis=1)

# 合并原始 + swapped
train_aug = pd.concat([base_train,swap_train],ignore_index=True)

print("Original training samples:", len(base_train))
print("Swapped training samples :", len(swap_train))
print("Final training samples   :", len(train_aug))


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

# Qwen 通常已经有 eos_token。
# 如果没有 pad_token，则使用 eos_token 作为 padding。
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# 分类任务不需要 left padding。
tokenizer.padding_side = "right"


print("Vocabulary size:", len(tokenizer))
print("PAD token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)


In [ ]:
train_dataset = Dataset.from_pandas(
    train_aug[["text","label"]],
    preserve_index=False,
)

val_dataset = Dataset.from_pandas(
    val_df[["text","label"]],
    preserve_index=False,
)

print(train_dataset)
print(val_dataset)


In [ ]:
def tokenize_function(examples):
    """
    将文本转换成 input_ids / attention_mask。

    truncation=True:
        如果超过 MAX_LENGTH，截断。

    max_length=MAX_LENGTH:
        控制最大 sequence length。

    对这个比赛来说非常重要，因为三个文本拼起来
    可能非常长。
    """

    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
)
val_dataset = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
)

print(train_dataset)
print(val_dataset)


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # NF4 是 QLoRA 中常用的 4-bit quantization 类型
    bnb_4bit_compute_dtype=torch.float16, # T4 / P100 使用 float16
    bnb_4bit_use_double_quant=True, # Double Quantization 可以进一步减少显存
)

# ============================================================
# Load Base Model
# ============================================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16, # `torch_dtype` is deprecated! Use `dtype` instead!
    trust_remote_code=True,
)

# ============================================================
# Model Configuration
# ============================================================

model.config.pad_token_id = tokenizer.pad_token_id
model.config.problem_type = ("single_label_classification")


In [ ]:
# 将量化模型转换成适合 k-bit training 的状态。
model = prepare_model_for_kbit_training(model)

# ============================================================
# LoRA Configuration
# ============================================================

lora_config = LoraConfig(
    # LoRA rank
    # r 越大：
    #   可训练参数越多
    #   表达能力更强
    #   显存/计算量也增加
    r=16,
    lora_alpha=32, # LoRA scaling
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ], # Qwen Attention 中最重要的 projection layers
    bias="none",
    task_type="SEQ_CLS", # 当前模型是 sequence classification
)

# ============================================================
# Attach LoRA Adapters
# ============================================================

model = get_peft_model(model,lora_config)
model.print_trainable_parameters()


In [ ]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
)


In [ ]:
def compute_metrics(eval_pred):
    """
    Trainer evaluation function。

    Kaggle 比赛真正关心的是：
        Log Loss

    同时输出：
        Accuracy
        Macro F1
    """

    logits, labels = eval_pred

    # logits -> probability
    probs = torch.softmax(torch.tensor(logits),dim=-1).numpy()
    predictions = probs.argmax(axis=1)

    result = {
        "log_loss": log_loss(
            labels,
            probs,
            labels=[0, 1, 2],
        ),
        "accuracy": accuracy_score(
            labels,
            predictions,
        ),
        "f1_macro": f1_score(
            labels,
            predictions,
            average="macro",
        ),
    }

    return result


In [ ]:
training_args = TrainingArguments(

    # 输出目录
    output_dir=OUTPUT_DIR,

    # Training
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_steps=0.05,
    # warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.

    # Mixed Precision
    fp16=True,

    # Gradient Checkpointing
    # 用计算换显存,对 T4/P100 很有用。
    gradient_checkpointing=True,

    # Optimizer
    # 8-bit AdamW 可以减少 optimizer memory。
    optim="paged_adamw_8bit",
    
    # Logging
    logging_steps=20,
    report_to="none",
    
    # Evaluation
    eval_strategy="steps",
    eval_steps=200,
    
    # Checkpoint
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    
    # Best Model
    load_best_model_at_end=True,
    metric_for_best_model="log_loss",
    greater_is_better=False,
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    # 新版 Transformers 使用 processing_class
    processing_class=tokenizer, # 不再使用 tokenizer=tokenizer
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer ready.")


In [ ]:
print("=" * 60)
print("START TRAINING")
print("=" * 60)

train_result = trainer.train()

print("=" * 60)
print("TRAINING FINISHED")
print("=" * 60)


In [ ]:
FINAL_MODEL_DIR = "/kaggle/working/final_qwen_qlora"

trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("Model saved to:")
print(FINAL_MODEL_DIR)


In [ ]:
eval_result = trainer.evaluate()

print("=" * 60)
print("VALIDATION RESULT")
print("=" * 60)

for key, value in eval_result.items():
    if isinstance(value, float):
        print(f"{key:25s}: {value:.6f}")
    else:
        print(f"{key:25s}: {value}")


In [ ]:
val_output = trainer.predict(val_dataset)
val_logits = val_output.predictions
val_probs = torch.softmax(torch.tensor(val_logits),dim=-1).numpy()
val_labels = val_df["label"].values

print("Validation probabilities shape:")
print(val_probs.shape)

print()
print("Validation Log Loss:")

print(
    log_loss(
        val_labels,
        val_probs,
        labels=[0, 1, 2]
    )
)


In [ ]:
val_pred = val_probs.argmax(axis=1)
cm = confusion_matrix(val_labels,val_pred,)
cm_df = pd.DataFrame(
    cm,
    index=[
        "True A",
        "True B",
        "True Tie"
    ],
    columns=[
        "Pred A",
        "Pred B",
        "Pred Tie"
    ],
)

display(cm_df)


In [ ]:
test_swap_df = test_df.copy()

# Swap A / B
test_swap_df["response_a"], test_swap_df["response_b"] = (
    test_df["response_b"].values,
    test_df["response_a"].values,
)

# 重新构造 text
test_swap_df["text"] = test_swap_df.apply(
    build_text,
    axis=1
)

print(test_swap_df["text"].iloc[0][:2000])


In [ ]:
test_swap_dataset = Dataset.from_pandas(
    test_swap_df[["text"]],
    preserve_index=False,
)
test_swap_dataset = test_swap_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
)

print(test_swap_dataset)


In [ ]:
test_output = trainer.predict(test_dataset)
test_logits = test_output.predictions
test_probs = torch.softmax(torch.tensor(test_logits),dim=-1).numpy()

swap_output = trainer.predict(test_swap_dataset)
swap_logits = swap_output.predictions
swap_probs = torch.softmax(torch.tensor(swap_logits),dim=-1).numpy()

print("Original probabilities:",test_probs.shape)
print("Swapped probabilities:",swap_probs.shape)

# ============================================================
# Convert Swapped Predictions Back
# ============================================================
swap_probs_corrected = swap_probs.copy()

# A probability becomes B
# B probability becomes A
swap_probs_corrected[:, 0] = swap_probs[:, 1]
swap_probs_corrected[:, 1] = swap_probs[:, 0]

# Tie stays unchanged
swap_probs_corrected[:, 2] = swap_probs[:, 2]


In [ ]:
final_probs = 0.5 * (test_probs + swap_probs_corrected)

# 防止极端情况下出现 0 probability。
EPS = 1e-7
final_probs = np.clip(final_probs, EPS, 1 - EPS,)

# 重新归一化
final_probs = final_probs / final_probs.sum(axis=1,keepdims=True)

print("Final probability shape:")
print(final_probs.shape)

print()
print("First 5 predictions:")

print(final_probs[:5])


In [ ]:
prob_sum = final_probs.sum(axis=1)

print("Min probability sum:", prob_sum.min())
print("Max probability sum:", prob_sum.max())
print(
    pd.DataFrame(
        final_probs,
        columns=[
            "winner_model_a",
            "winner_model_b",
            "winner_tie",
        ]
    ).head(10)
)


In [ ]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "winner_model_a": final_probs[:, 0],
    "winner_model_b": final_probs[:, 1],
    "winner_tie": final_probs[:, 2],
})

print("=" * 60)
print("SUBMISSION")
print("=" * 60)
print("Shape:", submission.shape)
display(submission.head(10))

required_columns = [
    "id",
    "winner_model_a",
    "winner_model_b",
    "winner_tie",
]

# 检查 columns
assert (
    submission.columns.tolist()
    == required_columns
)

# 检查 row 数
assert (
    len(submission)
    == len(test_df)
)

# 检查 ID
assert (
    submission["id"].equals(
        test_df["id"]
    )
)

# 检查概率
prob_columns = [
    "winner_model_a",
    "winner_model_b",
    "winner_tie",
]

# 所有概率必须 >= 0
assert (
    submission[prob_columns]
    .values
    >= 0
).all()

# 所有概率必须 <= 1
assert (
    submission[prob_columns]
    .values
    <= 1
).all()

# 每行概率和必须为 1
assert np.allclose(
    submission[prob_columns].sum(axis=1),
    1.0,
    atol=1e-5,
)

print("All submission checks passed.")


In [ ]:
SUBMISSION_PATH = "/kaggle/working/submission.csv"
submission.to_csv(SUBMISSION_PATH,index=False)

print("Submission saved:")
print(SUBMISSION_PATH)
display(pd.read_csv(SUBMISSION_PATH).head(20))
